In [ ]:
# Push dataset to Hugging Face Hub

# Prereqs:
# - pip install datasets huggingface_hub pandas tqdm
# - Set env var HUGGINGFACE_TOKEN or run huggingface-cli login in your environment

import os
import shutil
from pathlib import Path
import pandas as pd
from tqdm import tqdm

DATA_TABLE = Path("data/dataset_table.csv")  # source CSV with absolute paths
HF_DIR = Path("hf_dataset")                  # local folder to stage the dataset
AUDIO_DIR = HF_DIR / "audio"
REPO_ID = "your-username/your-dataset"       # TODO: set your repo id

HF_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

print(f"Staging to: {HF_DIR.resolve()}")



In [ ]:
# Load table and copy audio into repo-relative layout

df = pd.read_csv(DATA_TABLE)
print(df.head(2))

# Keep category/sample structure for clarity in the Hub repo
# Build relative paths under hf_dataset/audio/<category>/<sample>/

def path_parts(p: str):
    p = Path(p)
    # Expect .../data/<category>/<sample>/filename
    try:
        idx = p.parts.index("data")
        category = p.parts[idx+1]
        sample = p.parts[idx+2]
    except Exception:
        # Fallback: put flat under audio/misc
        category, sample = "misc", p.parent.name
    return category, sample, p.name

rel_targets = []
rel_options = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    # Target
    cat_t, samp_t, name_t = path_parts(row["AUDIO_TARGET"])
    dst_dir_t = AUDIO_DIR / cat_t / samp_t
    dst_dir_t.mkdir(parents=True, exist_ok=True)
    dst_path_t = dst_dir_t / name_t
    if not dst_path_t.exists():
        shutil.copy2(row["AUDIO_TARGET"], dst_path_t)
    rel_targets.append(dst_path_t.relative_to(HF_DIR).as_posix())

    # Options
    cat_o, samp_o, name_o = path_parts(row["AUDIO_OPTIONS"])
    dst_dir_o = AUDIO_DIR / cat_o / samp_o
    dst_dir_o.mkdir(parents=True, exist_ok=True)
    dst_path_o = dst_dir_o / name_o
    if not dst_path_o.exists():
        shutil.copy2(row["AUDIO_OPTIONS"], dst_path_o)
    rel_options.append(dst_path_o.relative_to(HF_DIR).as_posix())

# Write staged CSV with relative paths
staged = df.copy()
staged["AUDIO_TARGET"] = rel_targets
staged["AUDIO_OPTIONS"] = rel_options
staged_path = HF_DIR / "data.csv"
staged.to_csv(staged_path, index=False)
print("Wrote:", staged_path)



In [ ]:
# Add a README (Dataset Card)

readme = f"""
# Audio Consistency Checks Dataset

This dataset contains audio-based ambiguity-resolution tasks across categories (e.g., speech_tempo, voice_quality).

- AUDIO_TARGET: path to the target audio (relative to repo root)
- AUDIO_OPTIONS: path to a stitched options audio (A/B/C segments with spoken separators)
- CORRECT_OPTION: one of "Option A", "Option B", "Option C"
- CATEGORY, GROUP_ID, ROW_ID and additional metadata columns

"""

(HF_DIR / "README.md").write_text(readme)
print("Wrote:", HF_DIR / "README.md")



In [ ]:
# Push folder to the Hub (files-only)

from huggingface_hub import create_repo, upload_folder

create_repo(REPO_ID, repo_type="dataset", exist_ok=True)
upload_folder(folder_path=str(HF_DIR), repo_id=REPO_ID, repo_type="dataset")
print("Uploaded folder to:", REPO_ID)



In [ ]:
# Optional: Load and cast audio locally, then push processed Dataset

from datasets import load_dataset, Audio

raw = load_dataset("csv", data_files=str(staged_path))
train = raw["train"]
train = train.cast_column("AUDIO_INPUT", Audio())
train = train.cast_column("AUDIO_COMPLETION", Audio())
print(train[0].keys())

train.push_to_hub(REPO_ID)
print("Pushed processed dataset to:", REPO_ID)

